# Mask R-CNN Instance Segmentation（v2）

相較 `instance_fixed (1).ipynb` 的改動：
1. **凍結 BatchNorm**：`fpn_v2` 用的是一般 `BatchNorm2d`，batch_size=2 統計量不穩，而且舊版算 val loss 時會用驗證集更新 BN running stats（資料洩漏）。
2. **用 COCO mask mAP 挑 checkpoint**，並在測試集上評估、列出每個類別的 AP。
3. **資料增強**（水平翻轉 + ColorJitter）+ **LR warmup / MultiStep 衰減**。
4. **Repeat Factor Sampling**：處理類別不平衡（fish 17k 個 vs reptiles 290 個）。
5. `box_detections_per_img=200`（資料中單張圖最多有 150 個物件，預設上限 100 不夠）。
6. 視覺化改成 GT / 預測並排。

## 設定

In [ ]:
import os, json, math, datetime, contextlib, io
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
import torchvision.transforms as T
from PIL import Image
from pycocotools import mask as maskUtils
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import matplotlib.pyplot as plt

image_dir = "Dataset/img"
annotation_dir = "Dataset/annotations"
train_json_path = os.path.join(annotation_dir, "multiclass_train.json")
test_json_path = os.path.join(annotation_dir, "multiclass_test.json")

CFG = {
    "batch_size": 2,
    "lr": 0.005,
    "momentum": 0.9,
    "weight_decay": 1e-4,
    "num_epochs": 12,
    "lr_milestones": [8, 11],   # 在這些 epoch 開始時 lr x0.1
    "warmup_iters": 500,
    "grad_clip": 10.0,
    "use_rfs": True,            # Repeat Factor Sampling
    "rfs_threshold": 0.1,
    "val_ratio": 0.1,
    "seed": 42,
    "ckpt_path": "best_maskrcnn_v2.pth",
}

with open(train_json_path) as f:
    data = json.load(f)
with open(test_json_path) as f:
    test_data = json.load(f)

NUM_CLASSES = len(data["categories"]) + 1   # +1 是背景
id_to_name = {0: "background", **{c["id"]: c["name"] for c in data["categories"]}}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(CFG["seed"])
print(f"device: {device}  NUM_CLASSES: {NUM_CLASSES}")

## Dataset

- `train=True` 時做水平翻轉（image / mask / box 一起翻）和 ColorJitter（只動顏色，標註不用改；水下影像色偏變化大，這個很有用）。
- `keep_empty=True` 時保留沒有標註的圖片：**評估時一定要保留**，否則模型在純背景圖上的誤判不會被算進 mAP。

In [ ]:
class UIIS10KInstanceDataset(Dataset):
    def __init__(self, coco, image_dir, train=False, keep_empty=False):
        self.image_dir = image_dir
        self.train = train
        self.color_jitter = T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02)

        self.anns_by_img = {}
        for ann in coco["annotations"]:
            self.anns_by_img.setdefault(ann["image_id"], []).append(ann)

        if keep_empty:
            self.images = list(coco["images"])
        else:
            self.images = [img for img in coco["images"] if img["id"] in self.anns_by_img]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        info = self.images[idx]
        img = Image.open(os.path.join(self.image_dir, info["file_name"])).convert("RGB")
        width, height = img.size

        boxes, labels, masks = [], [], []
        for ann in self.anns_by_img.get(info["id"], []):
            rles = maskUtils.frPyObjects(ann["segmentation"], height, width)
            m = maskUtils.decode(rles)
            if m.ndim == 3:
                m = m.max(axis=2)

            ys, xs = np.where(m > 0)
            if len(xs) == 0:
                continue
            x1, y1, x2, y2 = xs.min(), ys.min(), xs.max() + 1, ys.max() + 1
            if x2 <= x1 or y2 <= y1:
                continue

            boxes.append([x1, y1, x2, y2])
            labels.append(ann["category_id"])
            masks.append(m)

        if self.train:
            img = self.color_jitter(img)

        image = torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0

        if len(boxes) == 0:
            boxes_t = torch.zeros((0, 4), dtype=torch.float32)
            labels_t = torch.zeros((0,), dtype=torch.int64)
            masks_t = torch.zeros((0, height, width), dtype=torch.uint8)
        else:
            boxes_t = torch.tensor(boxes, dtype=torch.float32)
            labels_t = torch.tensor(labels, dtype=torch.int64)
            masks_t = torch.from_numpy(np.stack(masks)).to(torch.uint8)

        # 水平翻轉（用 torch 的 RNG，DataLoader 的每個 worker 會自動拿到不同 seed）
        if self.train and torch.rand(1).item() < 0.5:
            image = image.flip(-1)
            masks_t = masks_t.flip(-1)
            boxes_t = boxes_t[:, [2, 1, 0, 3]] * torch.tensor([-1, 1, -1, 1]) + torch.tensor([width, 0, width, 0])

        target = {
            "boxes": boxes_t,
            "labels": labels_t,
            "masks": masks_t,
            "image_id": torch.tensor([info["id"]]),
        }
        return image, target


def collate_fn(batch):
    return tuple(zip(*batch))

## 切 train / val，建立 DataLoader

train 和 val 用**兩個不同的 Dataset 物件**（一個有增強、一個沒有），再用同一組 index 切開，切法跟舊版一致（seed 42、10%）。

In [ ]:
train_full = UIIS10KInstanceDataset(data, image_dir, train=True)
eval_full = UIIS10KInstanceDataset(data, image_dir, train=False)
assert [i["id"] for i in train_full.images] == [i["id"] for i in eval_full.images]

perm = np.random.default_rng(CFG["seed"]).permutation(len(train_full))
n_val = int(len(perm) * CFG["val_ratio"])
val_idx, train_idx = perm[:n_val], perm[n_val:]

train_ds = Subset(train_full, train_idx)
val_ds = Subset(eval_full, val_idx)
val_img_ids = [eval_full.images[i]["id"] for i in val_idx]

test_ds = UIIS10KInstanceDataset(test_data, image_dir, train=False, keep_empty=True)

print(f"train: {len(train_ds)}  val: {len(val_ds)}  test: {len(test_ds)}")

### Repeat Factor Sampling（類別不平衡）

LVIS 論文的做法：類別 c 出現在 f_c 比例的圖片中，該類別的 repeat factor = max(1, sqrt(t / f_c))；每張圖取它包含的類別中最大的 factor 當抽樣權重。稀有類別（reptiles、arthropoda、robots…）的圖會被更常抽到，常見類別不受影響。

In [ ]:
def repeat_factors(dataset_full, indices, t):
    img_cats = []
    for i in indices:
        anns = dataset_full.anns_by_img.get(dataset_full.images[i]["id"], [])
        img_cats.append({a["category_id"] for a in anns})

    n = len(img_cats)
    freq = {}
    for cats in img_cats:
        for c in cats:
            freq[c] = freq.get(c, 0) + 1
    cat_rf = {c: max(1.0, math.sqrt(t / (k / n))) for c, k in freq.items()}
    img_rf = [max((cat_rf[c] for c in cats), default=1.0) for cats in img_cats]
    return img_rf, cat_rf


if CFG["use_rfs"]:
    img_rf, cat_rf = repeat_factors(train_full, train_idx, CFG["rfs_threshold"])
    for c in sorted(cat_rf):
        print(f"  {id_to_name[c]:>12s}: repeat factor {cat_rf[c]:.2f}")
    sampler = WeightedRandomSampler(img_rf, num_samples=len(train_ds), replacement=True)
else:
    sampler = None

train_loader = DataLoader(
    train_ds, batch_size=CFG["batch_size"], shuffle=(sampler is None), sampler=sampler,
    collate_fn=collate_fn, num_workers=4, pin_memory=True,
)
val_loader = DataLoader(
    val_ds, batch_size=CFG["batch_size"], shuffle=False,
    collate_fn=collate_fn, num_workers=4, pin_memory=True,
)
test_loader = DataLoader(
    test_ds, batch_size=CFG["batch_size"], shuffle=False,
    collate_fn=collate_fn, num_workers=4, pin_memory=True,
)

## 檢查增強：畫一張翻轉 + ColorJitter 後的訓練圖，確認 mask / box 有對齊

In [ ]:
def draw_target(ax, image, target, title=""):
    ax.imshow(image.permute(1, 2, 0).clamp(0, 1))
    cmap = plt.get_cmap("tab10")
    for box, label, mask in zip(target["boxes"], target["labels"], target["masks"]):
        color = cmap(int(label) % 10)
        x1, y1, x2, y2 = box.tolist()
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=2))
        ax.text(x1, y1 - 5, id_to_name.get(int(label), str(int(label))), color=color, fontsize=9, weight="bold")
        overlay = np.zeros((*mask.shape, 4))
        overlay[mask.numpy() > 0] = (*color[:3], 0.4)
        ax.imshow(overlay)
    ax.set_title(title)
    ax.axis("off")


torch.manual_seed(0)
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax in axes:
    img, tgt = train_ds[0]
    draw_target(ax, img, tgt, "augmented sample")
plt.tight_layout()
plt.show()

## 建立模型

- 預訓練：COCO
- `box_detections_per_img=200`
- `freeze_bn`：把所有 BatchNorm 固定成 eval 模式、參數不更新。之後**每次呼叫 `model.train()` 後都要再呼叫一次**，因為 `train()` 會把 BN 切回訓練模式。

In [ ]:
from torchvision.models.detection import maskrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor


def build_model(num_classes):
    model = maskrcnn_resnet50_fpn_v2(weights="DEFAULT", box_detections_per_img=200)

    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, 256, num_classes)
    return model


def freeze_bn(model):
    for m in model.modules():
        if isinstance(m, torch.nn.BatchNorm2d):
            m.eval()
            for p in m.parameters():
                p.requires_grad_(False)


def set_train_mode(model):
    model.train()
    freeze_bn(model)


def build_optimizer_and_scheduler(model, iters_per_epoch):
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=CFG["lr"], momentum=CFG["momentum"],
                                weight_decay=CFG["weight_decay"])

    warmup = CFG["warmup_iters"]
    milestones = [m * iters_per_epoch for m in CFG["lr_milestones"]]

    def lr_lambda(it):   # 以 iteration 為單位：先線性 warmup，之後階梯式衰減
        factor = 0.1 ** sum(it >= m for m in milestones)
        if it < warmup:
            factor *= 0.001 + (1 - 0.001) * it / warmup
        return factor

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    return optimizer, scheduler

## 訓練 / 評估函式

- `train_one_epoch`：scheduler 每個 iteration 呼叫一次；加了梯度裁剪，避免一開始 loss 爆掉。
- `evaluate_loss`：仍然要用 train 模式才拿得到 loss，但因為 BN 已凍結，**不會再用驗證資料更新 BN**。
- `evaluate_map`：用 pycocotools 算 COCO 的 segm / bbox mAP。

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler):
    set_train_mode(model)
    total_loss, n = 0.0, 0

    for images, targets in loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())

        if not torch.isfinite(loss):
            print("  !! non-finite loss, skip batch:", {k: v.item() for k, v in loss_dict.items()})
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            continue

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        n += 1

    return total_loss / max(n, 1)


@torch.no_grad()
def evaluate_loss(model, loader):
    set_train_mode(model)   # 拿 loss 必須是 train 模式；BN 已凍結，不會被驗證資料改動
    total_loss = 0.0
    for images, targets in loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        loss_dict = model(images, targets)
        total_loss += sum(loss_dict.values()).item()
    return total_loss / len(loader)


@torch.no_grad()
def predict_coco_results(model, loader, score_thr=0.05):
    model.eval()
    results = []
    for images, targets in loader:
        outputs = model([img.to(device) for img in images])
        for out, tgt in zip(outputs, targets):
            img_id = int(tgt["image_id"])
            keep = out["scores"] >= score_thr
            boxes = out["boxes"][keep].cpu()
            labels = out["labels"][keep].cpu()
            scores = out["scores"][keep].cpu()
            masks = (out["masks"][keep, 0] > 0.5).cpu().numpy().astype(np.uint8)

            for box, lab, sc, m in zip(boxes, labels, scores, masks):
                rle = maskUtils.encode(np.asfortranarray(m))
                rle["counts"] = rle["counts"].decode("ascii")
                x1, y1, x2, y2 = box.tolist()
                results.append({
                    "image_id": img_id, "category_id": int(lab), "score": float(sc),
                    "segmentation": rle, "bbox": [x1, y1, x2 - x1, y2 - y1],
                })
    return results


def coco_eval(coco_gt, results, img_ids, iou_type="segm", verbose=True):
    if len(results) == 0:
        print("no detections")
        return None
    with contextlib.redirect_stdout(io.StringIO()):
        coco_dt = coco_gt.loadRes(results)
    E = COCOeval(coco_gt, coco_dt, iouType=iou_type)
    E.params.imgIds = list(img_ids)
    if verbose:
        E.evaluate(); E.accumulate(); E.summarize()
    else:
        with contextlib.redirect_stdout(io.StringIO()):
            E.evaluate(); E.accumulate(); E.summarize()
    return E


def per_class_ap(E):
    # precision: [IoU thresholds, recall, class, area range, max dets]
    precision = E.eval["precision"]
    out = {}
    for k, cat_id in enumerate(E.params.catIds):
        p = precision[:, :, k, 0, -1]
        p = p[p > -1]
        out[id_to_name[cat_id]] = float(p.mean()) if p.size else float("nan")
    return out


with contextlib.redirect_stdout(io.StringIO()):
    coco_train_gt = COCO(train_json_path)   # val 是從 train json 切出來的，GT 要用這個
    coco_test_gt = COCO(test_json_path)

## 健康檢查：單一 batch 過擬合測試

確認資料流 + 增強 + 凍結 BN 之後，loss 仍然能正常下降。這裡用獨立的 `sanity_model`，不會影響正式訓練。

In [ ]:
sanity_model = build_model(NUM_CLASSES).to(device)
sanity_opt = torch.optim.SGD([p for p in sanity_model.parameters() if p.requires_grad],
                             lr=0.005, momentum=0.9)

images, targets = next(iter(DataLoader(val_ds, batch_size=2, collate_fn=collate_fn)))
images = [img.to(device) for img in images]
targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

set_train_mode(sanity_model)
for step in range(50):
    loss_dict = sanity_model(images, targets)
    loss = sum(loss_dict.values())
    sanity_opt.zero_grad(set_to_none=True)
    loss.backward()
    sanity_opt.step()
    if step % 10 == 0:
        print(step, {k: round(v.item(), 4) for k, v in loss_dict.items()})

del sanity_model, sanity_opt, images, targets
torch.cuda.empty_cache()

## 正式訓練

每個 epoch 算 train loss、val loss、**val mask mAP**，用 mask mAP 挑最佳 checkpoint（存到 `best_maskrcnn_v2.pth`，不覆蓋舊的 `best_maskrcnn.pth`）。

In [ ]:
model = build_model(NUM_CLASSES).to(device)
optimizer, scheduler = build_optimizer_and_scheduler(model, len(train_loader))

history = {"train_loss": [], "val_loss": [], "val_segm_map": [], "val_bbox_map": [], "lr": []}
best_map, best_epoch = -1.0, -1

for epoch in range(CFG["num_epochs"]):
    lr_now = optimizer.param_groups[0]["lr"]
    train_loss = train_one_epoch(model, train_loader, optimizer, scheduler)
    val_loss = evaluate_loss(model, val_loader)

    results = predict_coco_results(model, val_loader)
    E_segm = coco_eval(coco_train_gt, results, val_img_ids, "segm", verbose=False)
    E_bbox = coco_eval(coco_train_gt, results, val_img_ids, "bbox", verbose=False)
    segm_map = E_segm.stats[0] if E_segm else 0.0
    bbox_map = E_bbox.stats[0] if E_bbox else 0.0

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_segm_map"].append(segm_map)
    history["val_bbox_map"].append(bbox_map)
    history["lr"].append(lr_now)

    print(f"Epoch [{epoch+1}/{CFG['num_epochs']}] lr {lr_now:.5f} | "
          f"train loss {train_loss:.4f} | val loss {val_loss:.4f} | "
          f"val mask mAP {segm_map:.4f} | val box mAP {bbox_map:.4f}")

    if segm_map > best_map:
        best_map, best_epoch = segm_map, epoch + 1
        torch.save(model.state_dict(), CFG["ckpt_path"])
        print("  -> saved best model")

torch.save(model.state_dict(), CFG["ckpt_path"].replace(".pth", "_last.pth"))
print(f"best val mask mAP {best_map:.4f} @ epoch {best_epoch}")

## 畫曲線並存檔

In [ ]:
def new_run_dir(run_name=None):
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    name = f"{timestamp}_{run_name}" if run_name else timestamp
    run_dir = os.path.join("Result", name)
    os.makedirs(run_dir, exist_ok=True)
    return run_dir


run_dir = new_run_dir("maskrcnn_instance_v2")
print("本次結果存放於:", run_dir)

epochs = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(epochs, history["train_loss"], label="train loss")
axes[0].plot(epochs, history["val_loss"], label="val loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].set_title("Loss"); axes[0].legend()
axes[1].plot(epochs, history["val_segm_map"], label="val mask mAP")
axes[1].plot(epochs, history["val_bbox_map"], label="val box mAP")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("mAP@[.5:.95]"); axes[1].set_title("Validation mAP"); axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(run_dir, "curves.png"), dpi=150, bbox_inches="tight")
plt.show()

## 測試集評估

載入最佳權重，在測試集（包含沒有標註的圖片）上算完整的 COCO 指標，以及每個類別的 mask AP。

In [ ]:
model.load_state_dict(torch.load(CFG["ckpt_path"], map_location=device))

test_results = predict_coco_results(model, test_loader)
test_img_ids = [img["id"] for img in test_ds.images]

print("===== Test: segm =====")
E_test_segm = coco_eval(coco_test_gt, test_results, test_img_ids, "segm")
print("\n===== Test: bbox =====")
E_test_bbox = coco_eval(coco_test_gt, test_results, test_img_ids, "bbox")

test_class_ap = per_class_ap(E_test_segm)
print("\n===== Test: per-class mask AP =====")
for name, ap in sorted(test_class_ap.items(), key=lambda x: -x[1]):
    print(f"  {name:>12s}: {ap:.4f}")

STAT_NAMES = ["AP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large",
              "AR1", "AR10", "AR100", "AR_small", "AR_medium", "AR_large"]

with open(os.path.join(run_dir, "history.json"), "w") as f:
    json.dump({
        "history": history,
        "config": CFG,
        "best_epoch": best_epoch,
        "best_val_segm_map": best_map,
        "test_segm": dict(zip(STAT_NAMES, map(float, E_test_segm.stats))),
        "test_bbox": dict(zip(STAT_NAMES, map(float, E_test_bbox.stats))),
        "test_per_class_mask_ap": test_class_ap,
    }, f, indent=2)

## 視覺化：GT（左）vs 預測（右）

In [ ]:
@torch.no_grad()
def show_gt_vs_pred(dataset, n=3, score_threshold=0.5, seed=0, save_path=None):
    model.eval()
    idxs = np.random.default_rng(seed).choice(len(dataset), n, replace=False)
    fig, axes = plt.subplots(n, 2, figsize=(14, 6 * n))
    axes = np.atleast_2d(axes)

    for row, i in zip(axes, idxs):
        image, target = dataset[i]
        draw_target(row[0], image, target, f"GT: {len(target['labels'])} object(s)")

        pred = model([image.to(device)])[0]
        keep = pred["scores"] > score_threshold
        pred_target = {
            "boxes": pred["boxes"][keep].cpu(),
            "labels": pred["labels"][keep].cpu(),
            "masks": (pred["masks"][keep, 0] > 0.5).cpu().to(torch.uint8),
        }
        draw_target(row[1], image, pred_target, f"Pred: {int(keep.sum())} object(s)")
        for txt, sc in zip(row[1].texts, pred["scores"][keep].cpu()):
            txt.set_text(f"{txt.get_text()} {sc:.2f}")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


show_gt_vs_pred(val_ds, n=3, save_path=os.path.join(run_dir, "val_predictions.png"))
show_gt_vs_pred(test_ds, n=3, save_path=os.path.join(run_dir, "test_predictions.png"))